<a href="https://colab.research.google.com/github/AmmarOkla12772/Deliverable3/blob/main/ML_Balanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Block 1: Loading the dataset, preparing features, and balancing the training data

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# The dataset is loaded from the provided CSV file.
# low_memory=False prevents dtype inference issues when pandas reads large files.
data = '/content/sample_data/stats19_collision_final6.csv'
df = pd.read_csv(data, low_memory=False)

# The target variable represents accident severity (severe vs non-severe).
y = df['target']


# Some attributes showed extremely weak association with severity during earlier
# exploratory analysis. These variables are removed to reduce noise and prevent
# the models from learning irrelevant patterns.
drop_features = [
    'collision_index',
    'num_vehicle_types',
    'num_pedestrians',
    'road_surface_conditions',
    'weather_conditions'
]

df_model = df.drop(columns=drop_features)


# Certain time-related attributes were originally numeric but actually represent
# categorical information. Converting them to categories avoids misleading the
# models into interpreting them as continuous quantities.

def hour_to_bin(hour):
    if 0 <= hour <= 5:
        return 'night'
    elif 6 <= hour <= 11:
        return 'morning'
    elif 12 <= hour <= 17:
        return 'afternoon'
    else:
        return 'evening'


# Hour values are grouped into broader time-of-day categories to capture
# meaningful traffic patterns rather than exact hours.
df_model['hour_cat'] = df_model['hour'].apply(hour_to_bin)

# Day and month are also treated as categorical values rather than numeric scales.
df_model['day_of_week_cat'] = df_model['day_of_week'].astype(str)
df_model['month_cat'] = df_model['month'].astype(str)

# Speed limit behaves more like a set of discrete categories than a continuous
# variable, so it is converted to string form as well.
df_model['speed_limit_cat'] = df_model['speed_limit'].astype(str)


# The original numeric columns that were replaced by categorical versions
# are removed to avoid duplicate information.
df_model = df_model.drop(columns=['hour', 'day_of_week', 'month', 'speed_limit'])


# Lists are created to keep track of categorical and numeric predictors.
# These will later be handled differently during preprocessing.
categorical_features = [
    'hour_cat',
    'day_of_week_cat',
    'month_cat',
    'speed_limit_cat',
    'road_type',
    'junction_detail',
    'light_conditions',
    'rear_impact_flag',
    'urban_or_rural_area',
    'any_skidding',
    'any_left_carriageway'
]

numeric_features = [
    'med_driver_age',
    'vehicle_difference_coefficient',
    'number_of_vehicles'
]


# The predictor matrix and target vector are separated.
X = df_model.drop(columns=['target'])
y = df_model['target']


# The dataset is split into training and testing subsets.
# Stratification preserves the original class ratio in the test set.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# The training data originally contains far more non-severe accidents (class 0)
# than severe ones (class 1). To prevent the models from simply favoring the
# majority class, the non-severe cases are randomly undersampled.

X_train_0 = X_train[y_train == 0]
y_train_0 = y_train[y_train == 0]

X_train_1 = X_train[y_train == 1]
y_train_1 = y_train[y_train == 1]


# A subset of the majority class is selected so that both classes
# contain the same number of observations.
X_train_bal = resample(
    X_train_0,
    replace=False,
    n_samples=len(y_train_1),
    random_state=42
)

y_train_bal = resample(
    y_train_0,
    replace=False,
    n_samples=len(y_train_1),
    random_state=42
)


# The sampled majority class and the full minority class are combined
# to form the balanced training dataset.
X_train_bal = pd.concat([X_train_bal, X_train_1], axis=0)
y_train_bal = pd.concat([y_train_bal, y_train_1], axis=0)


print(
    f"Balanced train size {X_train_bal.shape[0]} "
    f"(0/1 = {sum(y_train_bal==0)}/{sum(y_train_bal==1)})"
)

Balanced train size 31328 (0/1 = 15664/15664)


In [4]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# The preprocessing step handles numeric and categorical attributes differently.
# Numeric features are standardized so that they share a similar scale. This is
# particularly important for distance-based and gradient-based models.
# Categorical variables are converted into one-hot encoded vectors so that models
# can treat each category as a separate binary feature. Dropping the first category
# avoids redundant columns and prevents perfect multicollinearity.

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
])


# This helper function trains a model using the preprocessing pipeline above and
# evaluates its performance on the test dataset. The same procedure is reused for
# all models so that results remain directly comparable.

def train_evaluate(model_name, model, X_train_data, y_train_data):
    """
    A pipeline is constructed that first applies preprocessing and then fits the
    classifier. The trained model is evaluated on the held-out test data using
    common classification metrics.
    """

    pipeline = Pipeline([
        ('preproc', preprocessor),
        ('clf', model)
    ])

    # The model is trained on the provided training subset.
    pipeline.fit(X_train_data, y_train_data)

    # Predictions are generated for the test dataset.
    y_pred = pipeline.predict(X_test)

    # Standard evaluation metrics are calculated.
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    # Results are printed in a simple format so the performance of different
    # models can be easily compared.
    print(model_name)
    print("Accuracy", round(acc, 4))
    print("Precision", round(prec, 4))
    print("Recall", round(rec, 4))
    print("F1 score", round(f1, 4))
    print()

    return pipeline, {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1
    }

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Logistic Regression is a linear baseline model. Balanced class weights
# ensure that the minority class is treated equally during optimization.
# A high max_iter ensures solver convergence given the feature space.

logreg_model = LogisticRegression(
    class_weight='balanced',
    max_iter=5000,
    random_state=42
)

logreg_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', logreg_model)
])

logreg_pipeline.fit(X_train_bal, y_train_bal)
print("Logistic Regression training complete")

y_pred_logreg = logreg_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred_logreg)
prec = precision_score(y_test, y_pred_logreg)
rec = recall_score(y_test, y_pred_logreg)
f1 = f1_score(y_test, y_pred_logreg)

print("Logistic Regression")
print("Accuracy", round(acc, 4))
print("Precision", round(prec, 4))
print("Recall", round(rec, 4))
print("F1 score", round(f1, 4))

print("\nClassification Report:\n", classification_report(y_test, y_pred_logreg))

# k-Nearest Neighbors classifies samples based on the labels of their
# closest neighbors. Five neighbors is used as a simple starting point.

knn_model = KNeighborsClassifier(n_neighbors=5)

knn_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', knn_model)
])

knn_pipeline.fit(X_train_bal, y_train_bal)
print("k-Nearest Neighbors training complete")

y_pred_knn = knn_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred_knn)
prec = precision_score(y_test, y_pred_knn)
rec = recall_score(y_test, y_pred_knn)
f1 = f1_score(y_test, y_pred_knn)

print("k-Nearest Neighbors")
print("Accuracy", round(acc, 4))
print("Precision", round(prec, 4))
print("Recall", round(rec, 4))
print("F1 score", round(f1, 4))

print("\nClassification Report:\n", classification_report(y_test, y_pred_knn))

# Linear Support Vector Machine attempts to find a hyperplane that
# separates classes while maximizing the margin. Balanced weights
# ensure the minority class remains influential during training.

linear_svc_model = LinearSVC(
    class_weight='balanced',
    max_iter=5000,
    random_state=42
)

linear_svc_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', linear_svc_model)
])

linear_svc_pipeline.fit(X_train_bal, y_train_bal)
print("Linear SVC training complete")

y_pred_svc = linear_svc_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred_svc)
prec = precision_score(y_test, y_pred_svc)
rec = recall_score(y_test, y_pred_svc)
f1 = f1_score(y_test, y_pred_svc)

print("Linear SVC")
print("Accuracy", round(acc, 4))
print("Precision", round(prec, 4))
print("Recall", round(rec, 4))
print("F1 score", round(f1, 4))

print("\nClassification Report:\n", classification_report(y_test, y_pred_svc))

# Decision Tree partitions the feature space using hierarchical rules.
# No max_depth is set to allow learning complex relationships, though
# this can increase overfitting risk.

dt_model = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)

dt_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', dt_model)
])

dt_pipeline.fit(X_train_bal, y_train_bal)
print("Decision Tree training complete")

y_pred_dt = dt_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred_dt)
prec = precision_score(y_test, y_pred_dt)
rec = recall_score(y_test, y_pred_dt)
f1 = f1_score(y_test, y_pred_dt)

print("Decision Tree")
print("Accuracy", round(acc, 4))
print("Precision", round(prec, 4))
print("Recall", round(rec, 4))
print("F1 score", round(f1, 4))

print("\nClassification Report:\n", classification_report(y_test, y_pred_dt))

Logistic Regression training complete
Logistic Regression
Accuracy 0.5787
Precision 0.3429
Recall 0.6292
F1 score 0.4439

Classification Report:
               precision    recall  f1-score   support

           0       0.81      0.56      0.66     10738
           1       0.34      0.63      0.44      3916

    accuracy                           0.58     14654
   macro avg       0.57      0.59      0.55     14654
weighted avg       0.68      0.58      0.60     14654

k-Nearest Neighbors training complete
k-Nearest Neighbors
Accuracy 0.5742
Precision 0.3254
Recall 0.5531
F1 score 0.4098

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.58      0.67     10738
           1       0.33      0.55      0.41      3916

    accuracy                           0.57     14654
   macro avg       0.55      0.57      0.54     14654
weighted avg       0.66      0.57      0.60     14654

Linear SVC training complete
Linear SVC
Accuracy 0.578

In [13]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import numpy as np

# Subsample training data to reduce computation, since RBF SVM scales poorly with large datasets
subsample_idx = np.random.choice(len(X_train_bal), 10000, replace=False)
X_sub = X_train_bal.iloc[subsample_idx]
y_sub = y_train_bal.iloc[subsample_idx]

# RBF SVM with probability estimates and balanced class weights
rbf_svm_model = SVC(
    kernel='rbf',
    probability=True,
    class_weight='balanced',
    random_state=42
)

# Pipeline to ensure preprocessing is applied consistently
rbf_svm_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', rbf_svm_model)
])

# Model Training
rbf_svm_pipeline.fit(X_sub, y_sub)
print("RBF SVM training complete")

# Predict on test set
y_pred_rbf = rbf_svm_pipeline.predict(X_test)

# Computing standard classification metrics
acc = accuracy_score(y_test, y_pred_rbf)
prec = precision_score(y_test, y_pred_rbf)
rec = recall_score(y_test, y_pred_rbf)
f1 = f1_score(y_test, y_pred_rbf)

# Results
print("RBF SVM")
print("Accuracy", round(acc, 4))
print("Precision", round(prec, 4))
print("Recall", round(rec, 4))
print("F1 score", round(f1, 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rbf))

RBF SVM training complete
RBF SVM
Accuracy 0.5881
Precision 0.3548
Recall 0.6614
F1 score 0.4618

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.56      0.67     10738
           1       0.35      0.66      0.46      3916

    accuracy                           0.59     14654
   macro avg       0.59      0.61      0.56     14654
weighted avg       0.70      0.59      0.61     14654



In [9]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# A multi-layer perceptron (MLP) is a feedforward neural network capable of
# capturing complex nonlinear relationships in the data. The architecture
# consists of two hidden layers with 50 and 25 neurons, respectively.
# A higher iteration limit ensures the solver converges given the feature space.

mlp_model = MLPClassifier(
    hidden_layer_sizes=(50, 25),
    max_iter=300,
    random_state=42
)

# The pipeline combines preprocessing and model training. Preprocessing
# ensures that numerical features are scaled and categorical features are
# properly encoded, maintaining consistency across training and testing.

mlp_pipeline = Pipeline([
    ('preproc', preprocessor),
    ('clf', mlp_model)
])

# The model is trained on the balanced dataset to learn patterns for both
# classes while avoiding bias toward the majority class.

mlp_pipeline.fit(X_train_bal, y_train_bal)
print("MLP training complete")

# Predictions are generated on the untouched test set.

y_pred_mlp = mlp_pipeline.predict(X_test)

# Evaluation metrics provide insight into the model's ability to correctly
# classify both the majority and minority classes.

acc = accuracy_score(y_test, y_pred_mlp)
prec = precision_score(y_test, y_pred_mlp)
rec = recall_score(y_test, y_pred_mlp)
f1 = f1_score(y_test, y_pred_mlp)

print("MLP")
print("Accuracy", acc)
print("Precision", prec)
print("Recall", rec)
print("F1 score", f1)

# The classification report summarizes precision, recall, and F1 for each class.

print("\nClassification Report:\n", classification_report(y_test, y_pred_mlp))

MLP training complete
MLP
Accuracy 0.5990855739047359
Precision 0.33976770816293145
Recall 0.530388151174668
F1 score 0.4141988234121049

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.62      0.70     10738
           1       0.34      0.53      0.41      3916

    accuracy                           0.60     14654
   macro avg       0.56      0.58      0.55     14654
weighted avg       0.67      0.60      0.62     14654



In [10]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# The final MLP uses a slightly larger architecture with two hidden layers
# of 60 and 30 neurons. ReLU activation and Adam solver are chosen for
# efficient training. A small L2 regularization (alpha=0.001) helps
# reduce overfitting. The iteration limit is increased to 600 to ensure
# convergence.

mlp_final = MLPClassifier(
    hidden_layer_sizes=(60, 30),
    activation='relu',
    solver='adam',
    alpha=0.001,
    max_iter=600,
    random_state=42
)

# The pipeline combines preprocessing and the neural network, ensuring
# consistent transformations for both training and test data.

mlp_pipeline_final = Pipeline([
    ('preproc', preprocessor),
    ('clf', mlp_final)
])

# Training occurs on the balanced dataset to provide the model with
# sufficient representation of the minority class.

mlp_pipeline_final.fit(X_train_bal, y_train_bal)
print("Final MLP training complete")

# Predictions are made on the test set. Probabilities are computed
# to allow a custom threshold, which adjusts the precision-recall balance.

y_scores_final = mlp_pipeline_final.predict_proba(X_test)[:, 1]

# Applying a threshold of 0.47 for the positive class to control
# the trade-off between precision and recall (higher recall and lower precision expected here).

threshold = 0.47
y_pred_final = (y_scores_final >= threshold).astype(int)

# Evaluation metrics summarize the model's predictive performance
# for the binary classification task.

acc = accuracy_score(y_test, y_pred_final)
prec = precision_score(y_test, y_pred_final)
rec = recall_score(y_test, y_pred_final)
f1 = f1_score(y_test, y_pred_final)

print("Final Tuned MLP")
print("Accuracy", acc)
print("Precision", prec)
print("Recall", rec)
print("F1 score", f1)

# The classification report provides precision, recall, and F1 for each class.

print("\nClassification Report:\n", classification_report(y_test, y_pred_final))

Final MLP training complete
Final Tuned MLP
Accuracy 0.548723897911833
Precision 0.3231939163498099
Recall 0.6294688457609806
F1 score 0.4270986745213549

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.52      0.63     10738
           1       0.32      0.63      0.43      3916

    accuracy                           0.55     14654
   macro avg       0.56      0.57      0.53     14654
weighted avg       0.67      0.55      0.57     14654



In [12]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import pandas as pd

# Preprocessing and dimensionality reduction
# The preprocessor scales numerical features and encodes categorical features.
# PCA is applied to retain 90% of variance, reducing feature dimensionality
# while keeping most of the information. This speeds up clustering.

X_scaled = preprocessor.fit_transform(X_train_bal)
pca = PCA(n_components=0.9, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"Original features: {X_scaled.shape[1]}, Reduced via PCA: {X_pca.shape[1]}")

# Evaluate K-Means clustering for k = 2 to 6 using silhouette score

best_sil = -1
best_k = None

for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    print(f"k={k} -> silhouette={sil:.3f}")
    if sil > best_sil:
        best_sil = sil
        best_k = k
        best_labels = labels

print(f"\nBest k={best_k} with silhouette={best_sil:.3f}")

# Cluster analysis: calculate the proportion of severe accidents in each cluster
cluster_df = pd.DataFrame({
    'cluster': best_labels,
    'target': y_train_bal.reset_index(drop=True)
})

print("\nProportion of severe accidents per cluster:")
print(cluster_df.groupby('cluster')['target'].mean())

print("\nCluster sizes:")
print(cluster_df['cluster'].value_counts())

Original features: 43, Reduced via PCA: 25
k=2 -> silhouette=0.153
k=3 -> silhouette=0.132
k=4 -> silhouette=0.136
k=5 -> silhouette=0.112
k=6 -> silhouette=0.103

Best k=2 with silhouette=0.153

Proportion of severe accidents per cluster:
cluster
0    0.522156
1    0.486442
Name: target, dtype: float64

Cluster sizes:
cluster
1    19435
0    11893
Name: count, dtype: int64
